# Review Efficiency: Confidence-Guided vs. Random Sampling

Visualizing how much faster you find errors when you review lowest-confidence fields first.

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np

random.seed(42)
np.random.seed(42)

## 1. Simulate a Dataset

1000 field extractions. 15% error rate. Well-calibrated confidence: errors tend to have lower confidence, correct fields tend to have higher confidence, but with realistic overlap.

In [ ]:
N = 1000
ERROR_RATE = 0.15
n_errors = int(N * ERROR_RATE)
n_correct = N - n_errors

# Well-calibrated: errors get lower confidence (beta distribution skewed low)
# Correct fields get higher confidence (beta distribution skewed high)
# The distributions overlap in the 0.3-0.7 range for realism
error_confidences = np.random.beta(2, 5, size=n_errors)      # mean ~0.29
correct_confidences = np.random.beta(5, 2, size=n_correct)    # mean ~0.71

# Combine: (is_error, confidence)
fields = [(True, c) for c in error_confidences] + [(False, c) for c in correct_confidences]
random.shuffle(fields)  # shuffle to simulate real unordered data

print(f"Total fields: {N}")
print(f"Errors: {n_errors} ({ERROR_RATE:.0%})")
print(f"Error confidence: mean={error_confidences.mean():.2f}, std={error_confidences.std():.2f}")
print(f"Correct confidence: mean={correct_confidences.mean():.2f}, std={correct_confidences.std():.2f}")

## 2. Build the Curves

- **Confidence-guided**: sort by confidence ascending, review in that order
- **Random**: review in random order (expected: linear diagonal)
- **Perfect**: all errors are at the bottom (best possible)

In [ ]:
# Sort by confidence ascending (lowest first)
sorted_by_conf = sorted(fields, key=lambda x: x[1])

# Cumulative errors found as we review from lowest confidence to highest
cum_errors_guided = np.cumsum([1 if is_err else 0 for is_err, _ in sorted_by_conf])

# Normalize: fraction of total errors found
total_errors = n_errors
frac_errors_guided = cum_errors_guided / total_errors

# X axis: fraction of fields reviewed
frac_reviewed = np.arange(1, N + 1) / N

# Random baseline: linear
frac_errors_random = frac_reviewed

# Perfect: find all errors first, then correct fields
perfect = np.minimum(np.arange(1, N + 1) / total_errors, 1.0)

print(f"After reviewing 10% of fields ({int(N*0.1)} fields):")
idx_10 = int(N * 0.1) - 1
print(f"  Confidence-guided: {frac_errors_guided[idx_10]:.0%} of errors found")
print(f"  Random:            {frac_errors_random[idx_10]:.0%} of errors found")
print(f"  Perfect:           {min(perfect[idx_10], 1.0):.0%} of errors found")

## 3. Plot: Cumulative Error Discovery

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

ax.plot(frac_reviewed * 100, frac_errors_guided * 100, 'b-', linewidth=2, label='Confidence-guided')
ax.plot(frac_reviewed * 100, frac_errors_random * 100, 'k--', linewidth=1.5, label='Random sampling')
ax.plot(frac_reviewed * 100, np.minimum(perfect, 1.0) * 100, 'g:', linewidth=1.5, label='Perfect confidence')

# Shade the gain area between guided and random
ax.fill_between(frac_reviewed * 100, frac_errors_random * 100, frac_errors_guided * 100,
                alpha=0.15, color='blue', label='Gain over random')

# Mark the error-rate operating point
er_idx = int(N * ERROR_RATE) - 1
ax.axvline(x=ERROR_RATE * 100, color='red', linestyle=':', alpha=0.7)
ax.plot(ERROR_RATE * 100, frac_errors_guided[er_idx] * 100, 'ro', markersize=8)
ax.annotate(f'Review {ERROR_RATE:.0%} of fields\n'
            f'Find {frac_errors_guided[er_idx]:.0%} of errors\n'
            f'({frac_errors_guided[er_idx]/ERROR_RATE:.1f}x gain)',
            xy=(ERROR_RATE * 100, frac_errors_guided[er_idx] * 100),
            xytext=(ERROR_RATE * 100 + 15, frac_errors_guided[er_idx] * 100 - 15),
            fontsize=10, color='red',
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

ax.set_xlabel('Fields Reviewed (%)', fontsize=12)
ax.set_ylabel('Errors Found (%)', fontsize=12)
ax.set_title('Confidence-Guided Review vs. Random Sampling', fontsize=14)
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim(0, 100)
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. The Business Metric

At the error-rate operating point: review N fields (where N = error_rate * total), count how many errors you find guided vs. random.

In [ ]:
n_to_review = int(N * ERROR_RATE)
errors_found_guided = cum_errors_guided[n_to_review - 1]
errors_found_random = ERROR_RATE * n_to_review  # expected value

gain = errors_found_guided / errors_found_random

print(f"Fields reviewed:          {n_to_review} ({ERROR_RATE:.0%} of {N})")
print(f"Errors found (guided):    {errors_found_guided}")
print(f"Errors found (random):    {errors_found_random:.0f}")
print(f"")
print(f"Gain: {gain:.1f}x")
print(f"")
print(f'"Confidence-guided review finds errors {gain:.1f}x faster than random sampling."')

## 5. Hours Saved (Example)

If each field review takes 30 seconds, how much time to catch 90% of errors?

In [ ]:
SECONDS_PER_REVIEW = 30
TARGET_CAPTURE = 0.90

# How many fields to review to catch 90% of errors?
target_errors = int(total_errors * TARGET_CAPTURE)

# Guided: find the index where we hit the target
fields_to_review_guided = np.searchsorted(cum_errors_guided, target_errors) + 1

# Random: need to review 90% of fields to catch 90% of errors
fields_to_review_random = int(N * TARGET_CAPTURE)

hours_guided = (fields_to_review_guided * SECONDS_PER_REVIEW) / 3600
hours_random = (fields_to_review_random * SECONDS_PER_REVIEW) / 3600

print(f"To catch {TARGET_CAPTURE:.0%} of errors ({target_errors} errors):")
print(f"  Confidence-guided: review {fields_to_review_guided} fields ({hours_guided:.1f} hours)")
print(f"  Random sampling:   review {fields_to_review_random} fields ({hours_random:.1f} hours)")
print(f"  Time saved:        {hours_random - hours_guided:.1f} hours ({(1 - hours_guided/hours_random):.0%} reduction)")

## 6. What Bad Confidence Looks Like

In [ ]:
# Poorly calibrated: confidence is random, no correlation with correctness
bad_fields = [(is_err, random.random()) for is_err, _ in fields]
bad_sorted = sorted(bad_fields, key=lambda x: x[1])
cum_errors_bad = np.cumsum([1 if is_err else 0 for is_err, _ in bad_sorted])
frac_errors_bad = cum_errors_bad / total_errors

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.plot(frac_reviewed * 100, frac_errors_guided * 100, 'b-', linewidth=2, label='Well-calibrated')
ax.plot(frac_reviewed * 100, frac_errors_bad * 100, 'r-', linewidth=2, label='Poorly-calibrated')
ax.plot(frac_reviewed * 100, frac_errors_random * 100, 'k--', linewidth=1.5, label='Random')

ax.set_xlabel('Fields Reviewed (%)', fontsize=12)
ax.set_ylabel('Errors Found (%)', fontsize=12)
ax.set_title('Good vs. Bad Confidence Calibration', fontsize=14)
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim(0, 100)
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Gain for bad confidence
bad_gain = cum_errors_bad[n_to_review - 1] / errors_found_random
print(f"Well-calibrated gain: {gain:.1f}x")
print(f"Poorly-calibrated gain: {bad_gain:.1f}x")
print(f"Random gain: 1.0x")